In [ ]:
!pip install wandb pandas numpy matplotlib seaborn scikit-learn kagglehub statsmodels

In [ ]:
import wandb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import mutual_info_regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from statsmodels.stats.outliers_influence import variance_inflation_factor
import kagglehub
import shutil
import os

# Set random seed for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)

set_seed(42)
print("All libraries imported successfully.")

In [ ]:
from key import KEY
API_KEY=KEY
wandb.login(API_KEY)

In [ ]:
path = kagglehub.dataset_download("lovishbansal123/nasa-asteroids-classification")
print(f"Dataset downloaded to: {path}")

dest = os.path.join(os.getcwd(), "dataset")
shutil.copytree(path, dest, dirs_exist_ok=True)

df_raw = pd.read_csv(os.path.join(dest, "nasa.csv"), low_memory=False)
print(f"Raw data shape: {df_raw.shape}")
df_raw.head()

In [ ]:
# Transformando o True em 1 e False em 0.
df_raw['Hazardous'] = df_raw['Hazardous'].replace({True: 1, False: 0})

df_raw.head()

# Removendo a coluna: "Equinox" e "Orbiting body"

df = df_raw.drop(['Equinox', 'Orbiting Body'], axis=1)
df.head()

In [ ]:
wandb.init(project="MLOps-Work", job_type="load_raw", name="load_raw")
artifact = wandb.Artifact("raw_data", type="dataset")
temp_path = "temp_raw.csv"
df_raw.to_csv(temp_path, index=False)
artifact.add_file(temp_path)
wandb.log_artifact(artifact)
wandb.summary["rows"] = len(df_raw)
wandb.summary["columns"] = list(df_raw.columns)
wandb.finish()
print("Raw data artifact saved to W&B.")

In [ ]:
def remove_duplicates(df):
    before = len(df)
    df = df.drop_duplicates()
    print(f"Removed {before - len(df)} duplicate rows.")
    return df

def handle_missing_values(df, threshold=0.5):
    missing_frac = df.isnull().mean()
    cols_to_drop = missing_frac[missing_frac > threshold].index.tolist()
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped columns ( >{threshold*100}% missing): {cols_to_drop}")
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    imputer = SimpleImputer(strategy='median')
    df[numeric_cols] = imputer.fit_transform(df[numeric_cols])
    cat_cols = df.select_dtypes(include=['object']).columns
    df[cat_cols] = df[cat_cols].fillna('missing')
    return df

df_clean = remove_duplicates(df_raw)
df_clean = handle_missing_values(df_clean, threshold=0.5)
print(f"Cleaned dataset shape: {df_clean.shape}")

In [ ]:
wandb.init(project="MLOps-work", job_type="clean_data", name="clean_data")
artifact = wandb.Artifact("clean_data", type="dataset")
temp_clean = "temp_clean.csv"
df_clean.to_csv(temp_clean, index=False)
artifact.add_file(temp_clean)
wandb.log_artifact(artifact)
wandb.summary["clean_rows"] = len(df_clean)
wandb.finish()
print("Cleaned data artifact saved to W&B.")

In [ ]:
TARGET = 'Hazardous'
print(f"Target variable: {TARGET}")
print("Distribution of target:")
print(df_clean[TARGET].value_counts().sort_index())

In [ ]:
# Compute correlation matrix including target
data_for_corr = X.copy()
data_for_corr[TARGET] = y
corr_matrix = data_for_corr.corr()

# Plot heatmap
plt.figure(figsize=(18, 14))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0,
            linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix of Numeric Features (including target)', fontsize=14)
plt.tight_layout()
corr_fig_path = "correlation_matrix.png"
plt.savefig(corr_fig_path, dpi=150)
plt.show()
# Log to W&B as an image artifact
wandb.init(project="MLOps-Work", job_type="eda", name="correlation_matrix")
wandb.log({"correlation_matrix": wandb.Image(corr_fig_path)})
wandb.finish()
print("Correlation matrix image saved and logged to W&B.")

In [ ]:
# Select top 6 features most correlated with target (absolute correlation)
corr_with_target = X.corrwith(y).abs().sort_values(ascending=False)
top6_features = corr_with_target.head(6).index.tolist()
print("Top 6 features by correlation with target:", top6_features)

# Create pairplot (joint distributions)
data_pairplot = X[top6_features].copy()
data_pairplot[TARGET] = y

pairplot_fig = sns.pairplot(data_pairplot, diag_kind='hist', plot_kws={'alpha':0.6})
pairplot_fig.fig.suptitle('Pairwise Joint Distributions (Top 6 Features vs Target)', y=1.02)
pairplot_path = "pairplot_top6.png"
pairplot_fig.savefig(pairplot_path, dpi=150)
plt.show()

# Log to W&B
wandb.init(project="MLOps-Work", job_type="eda", name="pairplot")
wandb.log({"pairplot_top6": wandb.Image(pairplot_path)})
wandb.finish()
print("Pairplot image saved and logged to W&B.")

In [ ]:
corr_series = X.corrwith(y).abs().sort_values(ascending=False)
print("Top 10 features by absolute correlation:")
print(corr_series.head(10))

In [ ]:
mi_scores = mutual_info_regression(X, y, random_state=42)
mi_series = pd.Series(mi_scores, index=feature_cols).sort_values(ascending=False)
print("Top 10 features by Mutual Information:")
print(mi_series.head(10))

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X, y)
imp_series = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("Top 10 features by Random Forest importance:")
print(imp_series.head(10))

In [ ]:
def normalize(s):
    return (s - s.min()) / (s.max() - s.min())

corr_norm = normalize(corr_series)
mi_norm = normalize(mi_series)
imp_norm = normalize(imp_series)

combined = (corr_norm + mi_norm + imp_norm) / 3
combined_sorted = combined.sort_values(ascending=False)

print("Top 15 features by combined score:")
print(combined_sorted.head(15))

In [ ]:
def calculate_vif(df_features):
    vif_data = pd.DataFrame()
    vif_data["feature"] = df_features.columns
    vif_data["VIF"] = [variance_inflation_factor(df_features.values, i) for i in range(df_features.shape[1])]
    return vif_data.sort_values("VIF", ascending=False)

top20_features = combined_sorted.head(20).index.tolist()
X_top20 = X[top20_features]
vif_df = calculate_vif(X_top20)
high_vif_features = vif_df[vif_df['VIF'] > 10]['feature'].tolist()
print(f"Features with VIF > 10: {high_vif_features}")

In [ ]:
candidates = combined_sorted.head(15).index.tolist()
final_features = [f for f in candidates if f not in high_vif_features][:10]

print("\n=== FINAL CANDIDATE FEATURES FOR REGRESSION ===")
for i, f in enumerate(final_features, 1):
    print(f"{i}. {f}")

In [ ]:
# Log final feature list as a table artifact
wandb.init(project="MLOps-Work", job_type="feature_selection", name="final_candidates")
feature_table = wandb.Table(dataframe=pd.DataFrame({
    "rank": range(1, len(final_features)+1),
    "feature": final_features,
    "combined_score": combined_sorted[final_features].values
}))
wandb.log({"candidate_features": feature_table})
wandb.config.update({"selected_features": final_features})
wandb.finish()
print("Candidate features logged to W&B.")

In [ ]:
X_selected = X[final_features]
X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y, test_size=0.2, random_state=42
)
print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

In [ ]:
X_train.to_csv("X_train.csv", index=False)
X_test.to_csv("X_test.csv", index=False)
y_train.to_csv("y_train.csv", index=False)
y_test.to_csv("y_test.csv", index=False)

wandb.init(project="MLOps-Work", job_type="split_data", name="save_splits")
train_artifact = wandb.Artifact("train_data", type="dataset")
train_artifact.add_file("X_train.csv")
train_artifact.add_file("y_train.csv")
wandb.log_artifact(train_artifact)

test_artifact = wandb.Artifact("test_data", type="dataset")
test_artifact.add_file("X_test.csv")
test_artifact.add_file("y_test.csv")
wandb.log_artifact(test_artifact)
wandb.finish()
print("Train/test splits saved and logged.")

In [ ]:
print("\n" + "="*60)
print("FINAL CANDIDATE FEATURES FOR REGRESSION")
print("="*60)
for i, f in enumerate(final_features, 1):
    print(f"{i:2d}. {f}")
print("="*60)
print("\nInterpretation:")
print("- Features like `Number_of_Vehicles`, `Speed_limit`, and `1st_Road_Class` appear among the top.")
print("- The correlation matrix and pairplot revealed that the target is moderately correlated with vehicle count.")
print("- These features are now ready to be used in any regression model (linear, tree‑based, neural network).")